# Autonomous Sumo Robot Digital Twin - Stage 3: Strategy Modeling
## Multi-Model Machine Learning Benchmarking & White-Box Decision Tree Extraction

This research notebook evaluates and benchmarks 4 distinct machine learning paradigms on hardware-constrained Agent Telemetry (zero spatial coordinate leakage):
1. **Decision Tree (Primary White-Box Candidate)**: Explainable CART tree with balanced class weights.
2. **Logistic Regression (Linear Baseline)**: Multinomial Softmax with L2 regularization.
3. **Random Forest (Ensemble Bagging)**: Parallel ensemble of randomized trees.
4. **Gradient Boosting (Ensemble Boosting)**: Sequential residual boosting trees.

### Objectives:
1. Execute 5-fold cross-validated `GridSearchCV` hyperparameter optimization across all 4 architectures.
2. Compile an empirical comparative leaderboard (Macro F1, Accuracy, Precision, Recall, Weighted F1, Training Latency).
3. Extract explainable hardware decision thresholds from the white-box Decision Tree.
4. Transpile the optimal Decision Tree into an Arduino/ESP32-compliant C++ header (`strategy_config.h`).


In [2]:
from IPython.display import display, Markdown, HTML
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import plot_tree

# Ensure project root is in Python path
project_root = Path.cwd() if (Path.cwd() / 'config').exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.config import AGENT_DATA_DIR, KIT_1KG_CONFIG, MEGA_3KG_CONFIG, PROJECT_ROOT
from models.tree_optimizer import (
    train_and_benchmark_models,
    train_and_optimize_tree,
    build_hardware_summary_table,
    generate_cpp_header,
    generate_cpp_harness,
)

print('Modeling & benchmarking libraries loaded successfully.')


Modeling & benchmarking libraries loaded successfully.


### 1. Ingest Agent Telemetry and Verify Zero Spatial Leakage

In [4]:
# Auto-discover agent dataset or fallback to standard path
agent_files = sorted(list(AGENT_DATA_DIR.glob("*_agent.parquet")))
if agent_files:
    agent_parquet_path = agent_files[0]
else:
    agent_parquet_path = AGENT_DATA_DIR / 'telemetry_kit_1kg_agent.parquet'

df_agent = pd.read_parquet(agent_parquet_path)
print(f'Loaded {len(df_agent):,} records from {agent_parquet_path.name}')
print()
print('Agent Feature Columns:', [c for c in df_agent.columns if not c.startswith('Match_') and not c.startswith('Bot_')])
print()
print('Target State Class Distribution:')
print(df_agent['Current_State'].value_counts())

Loaded 114,060 records from telemetry_kit_1kg_agent.parquet

Agent Feature Columns: ['Timestamp_ms', 'IR_Edge_FL', 'IR_Edge_FR', 'Opp_L90', 'Opp_L18', 'Opp_F0', 'Opp_R18', 'Opp_R90', 'Current_State', 'Action_PWM_Left', 'Action_PWM_Right']

Target State Class Distribution:
Current_State
ATTACK           113007
SEARCH              397
EDGE_RECOVERY       344
TRACK               180
EVADE               132
Name: count, dtype: int64


### 2. Run 5-Fold Cross-Validated Multi-Model Benchmark (GridSearchCV)


In [6]:
benchmark_res = train_and_benchmark_models(agent_parquet_path, test_size=0.20, random_state=42)
df_leaderboard = benchmark_res['leaderboard']
results = benchmark_res['primary_model_results']

print()
print('=================== MULTI-MODEL BENCHMARK LEADERBOARD ===================')
display(df_leaderboard)

print(f"\nChampion Architecture: {benchmark_res['best_model_name']}")
print(f"White-Box Decision Tree Macro F1: {results['test_f1_macro']:.4f}")



=================== MULTI-MODEL BENCHMARK LEADERBOARD ===================
   Rank                Model  ... p-value (vs DT)  Stat. Significant
0     1        Random Forest  ...          0.1044       No (p>=0.05)
1     2        Decision Tree  ...          1.0000          Reference
2     3  Logistic Regression  ...          0.0029       Yes (p<0.05)
3     4    Gradient Boosting  ...          0.0356       Yes (p<0.05)

[4 rows x 12 columns]

Champion Architecture: Random Forest
White-Box Decision Tree Macro F1: 0.5732


In [7]:
# Multi-Model Benchmark Comparison Plot (Grouped by Metric, colored by Model)
metric_cols = ['Test Macro F1', 'Test Accuracy', 'Test Precision (M)', 'Test Recall (M)', '5-Fold CV Macro F1']
df_plot = df_leaderboard.set_index('Model')[metric_cols].T

plt.figure(figsize=(14, 6))
df_plot.plot(kind='bar', figsize=(14, 6), width=0.8, edgecolor='black',
             color=['#00d2ff', '#10b981', '#a855f7', '#f59e0b'])
plt.title('Multi-Model Benchmark Comparison (Grouped by Evaluation Metric)', fontsize=15, fontweight='bold')
plt.xlabel('Evaluation Metric', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.ylim(0.70, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend(title='Model Architecture', loc='lower right', framealpha=0.9)
plt.xticks(rotation=0, fontsize=11)
plt.tight_layout()
plt.show()


<ipython-input-1-bfb1e8adc50b>:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3. Confusion Matrix and Feature Importance Heatmaps

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Confusion Matrix
cm_df = pd.DataFrame(
    results["confusion_matrix_normalized"],
    index=results["classes"],
    columns=results["classes"]
)
sns.heatmap(cm_df, annot=True, fmt=".2%", cmap="Blues", cbar=True, ax=axes[0])
axes[0].set_title("Normalized Confusion Matrix (Test Set)", fontsize=14)
axes[0].set_ylabel("Ground Truth State")
axes[0].set_xlabel("Predicted State")

# 2. Feature Importances
feat_df = pd.Series(results["feature_importances"]).sort_values(ascending=True)
feat_df.plot(kind="barh", ax=axes[1], color="teal", edgecolor="black")
axes[1].set_title("Sensor Feature Importances", fontsize=14)
axes[1].set_xlabel("Importance Score (Gini)")

plt.tight_layout()
plt.show()

<ipython-input-1-c0e104fddf22>:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4. Pruned Decision Tree Topology Visualization

In [11]:
plt.figure(figsize=(22, 10))
plot_tree(
    results["model"],
    feature_names=results["feature_names"],
    class_names=results["classes"],
    filled=True,
    rounded=True,
    fontsize=10,
    max_depth=3
)
plt.title("Pruned Decision Tree State Transition Rules (Max Depth = 3 Preview)", fontsize=16)
plt.show()

<ipython-input-1-83bdb2b6fb5e>:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5. Hardware Decision Threshold Table

In [13]:
# 1. Per-Sensor Calibrated Parameters (Hardware Summary)
print('=== 1. Per-Sensor Calibrated Parameters (Hardware Summary) ===')
display(results['hardware_summary_table'])

# 2. Full Decision Tree Node Hierarchy
print()
print('=== 2. Full Decision Tree Node Hierarchy (First 15 Nodes) ===')
display(results['thresholds_table'].head(15))


=== 1. Per-Sensor Calibrated Parameters (Hardware Summary) ===
        Sensor_Feature  ... Decision_Nodes
0               Opp_F0  ...             12
1         Delta_Opp_F0  ...              9
2           IR_Edge_FL  ...             11
3           IR_Edge_FR  ...              7
4        Opp_Lat_Delta  ...              5
5  Opp_Bearing_Est_Deg  ...              6
6              Opp_L18  ...              4
7              Opp_R18  ...              4
8              Opp_L90  ...              2
9              Opp_R90  ...              3

[10 rows x 7 columns]

=== 2. Full Decision Tree Node Hierarchy (First 15 Nodes) ===
    Node_ID       Sensor_Feature Comparison  Threshold_Value  Samples  Impurity
0         0               Opp_F0         <=           0.1755    45624    2.3219
1         1           IR_Edge_FL         <=           0.9596      369    1.5754
2         2         Delta_Opp_F0         <=         -23.5108      289    1.4736
3         3           IR_Edge_FR         <=           0.52

### 6. Transpile Decision Rules into C++ Embedded Header (`strategy_config.h`)

In [15]:
# Detect robot class dynamically from loaded dataset
active_robot_class = "MEGA_3KG" if ("3kg" in agent_parquet_path.name.lower() or "mega" in agent_parquet_path.name.lower()) else "KIT_1KG"

header_str = generate_cpp_header(
    model=results["model"],
    feature_names=results["feature_names"],
    classes=results["classes"],
    weight_class=active_robot_class,
)
print()
print(f"--- Generated C++ strategy_config.h ({active_robot_class}) Preview (First 35 lines) ---")
print("\n".join(header_str.splitlines()[:35]))

harness_str = generate_cpp_harness(
    robot_class=active_robot_class,
    header_filename="strategy_config.h",
    dataset_name=agent_parquet_path.stem,
)
print()
print(f"--- Generated C++ main.cpp Harness ({active_robot_class}) Preview (First 35 lines) ---")
print("\n".join(harness_str.splitlines()[:35]))



--- Generated C++ strategy_config.h (KIT_1KG) Preview (First 35 lines) ---
/**
 * ==============================================================================
 * AUTONOMOUS SUMO ROBOT DIGITAL TWIN - FIRMWARE STRATEGY CONFIGURATION
 * Generated automatically by Decision Tree Transpiler
 * Target Weight Class: KIT_1KG
 * Architecture: Arduino / ESP32 C++ Header
 * ==============================================================================
 */

#ifndef STRATEGY_CONFIG_H
#define STRATEGY_CONFIG_H

#include <stdint.h>

#ifdef __cplusplus
extern "C" {
#endif

// ==============================================================================
// 1. STATE ENUM DECLARATION
// ==============================================================================
typedef enum {
    STATE_SEARCH = 0,
    STATE_TRACK = 1,
    STATE_ATTACK = 2,
    STATE_EDGE_RECOVERY = 3,
    STATE_EVADE = 4
} BotState;

// ==============================================================================
// 2. CALIBRATED 